[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qualia906/Developing-Agentic-AI-with-LangChain/blob/main/chap02/hands-on/chap02_hands_on.ipynb)


# 第2章 ハンズオン：LangChain エージェントの基本構築

## 学習目標
- `init_chat_model` を使ってLLMを初期化する
- `create_agent` を使ってエージェントを作成する
- エージェントに質問を投げかけ、応答を受け取る

---

## Step 1: ライブラリのインストール

まず必要なライブラリをインストールします。

In [ ]:
# Google Colab の場合はこのセルを実行してインストールします
%pip install -qU langchain langchain-openai langgraph

## Step 2: 環境変数の設定

LangSmith によるトレース（実行の可視化）と OpenAI API を使用するために、
API キーを環境変数として設定します。

In [ ]:
import os
from google.colab import userdata

# LangSmith によるトレースを有効にする
# LangSmith: https://smith.langchain.com/ でトレースが確認できます
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap02-handson"

# OpenAI API キー
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("環境変数の設定が完了しました")

## Step 3: LLMの初期化

`init_chat_model` は、モデル名のプレフィックス（例: `openai:`）から
自動的に適切なプロバイダーを選択してくれる便利な関数です。

In [ ]:
from langchain.chat_models import init_chat_model

# 「プロバイダー:モデル名」の形式で指定するだけでOK
# openai:gpt-4o → OpenAI の GPT-4o を使用
llm = init_chat_model("openai:gpt-4o")

# LLM 単体でもメッセージを送れます（エージェントとの違いに注目）
response = llm.invoke("こんにちは！自己紹介をしてください。")
print("LLM の応答:", response.content)

## Step 4: エージェントの作成

`create_agent` は LangChain v1.0 の標準的なエージェント構築方法です。
内部では LangGraph を使い、ツール呼び出しのループを自動で管理してくれます。

**LLM 単体 vs エージェントの違い:**
- LLM 単体: 1回の推論で返答する（ループしない）
- エージェント: ツールを活用しながら、目的達成まで繰り返し推論できる

In [ ]:
from langchain.agents import create_agent

# create_agent でエージェントを作成
# tools=[] → ツールなし（この章では LLM の会話能力のみを使う）
agent = create_agent(
    model="openai:gpt-4o",     # 使用するモデルを指定
    tools=[],                   # 使用ツールのリスト（今は空）
    system_prompt=(             # エージェントの役割定義（システムプロンプト）
        "あなたは親切で知識豊富なAIアシスタントです。"
        "日本語で丁寧に回答してください。"
    ),
)

print("エージェントの作成が完了しました")
print(f"エージェントの型: {type(agent)}")

## Step 5: エージェントに質問する

`agent.invoke()` でエージェントを呼び出します。
入力は `{"messages": [...]}` の形式で渡します。

In [ ]:
# エージェントにメッセージを投げかける
# messages は OpenAI の Chat Completions API と同じ形式
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "LangChain とは何ですか？3行で簡潔に説明してください。"
        }
    ]
})

# 最後のメッセージ（エージェントの返答）を取得
final_message = result["messages"][-1]
print("エージェントの回答:")
print(final_message.content)

## Step 6: 応答のメッセージ構造を確認する

エージェントの結果には複数のメッセージが含まれています。
LangSmith のトレースと合わせて確認してみましょう。

In [ ]:
# メッセージの一覧を確認する（ツールなしなので2つのみ）
print(f"メッセージ数: {len(result['messages'])}")
print()

for i, msg in enumerate(result["messages"]):
    # メッセージの種類（HumanMessage / AIMessage）を表示
    msg_type = type(msg).__name__
    print(f"[メッセージ {i+1}] 種別: {msg_type}")
    print(f"  内容: {msg.content[:100]}..." if len(msg.content) > 100 else f"  内容: {msg.content}")
    print()

## Step 7: 様々な質問を試してみよう

自由に質問を変えて、エージェントの応答を確認してみましょう。
LangSmith のダッシュボード（https://smith.langchain.com/）でトレースも確認できます。

In [ ]:
# 好きな質問に変えて試してみてください
my_question = "AIの最新トレンドを3つ挙げてください。"



In [ ]:
result2 = agent.invoke({
    "messages": [
        {"role": "user", "content": my_question}
    ]
})

print(f"質問: {my_question}")
print()
print("回答:")
print(result2["messages"][-1].content)

## 【オプション】対話的な実行

`input()` 関数を使って、Google Colab 上で直接プロンプトを入力してエージェントを実行してみましょう。

In [ ]:
# Pythonの input() を使って対話的に質問を入力します
# 実行すると入力欄が表示されるので、質問を入力してEnterを押してください
interactive_input = input("エージェントへの入力: ")

In [ ]:
# 入力された質問でエージェントを実行します
result_interactive = agent.invoke({
    "messages": [
        {"role": "user", "content": interactive_input}
    ]
})

print("回答:")
print(result_interactive["messages"][-1].content)

---

## まとめ

この章では以下を学びました：

| 概念 | 使用した API | ポイント |
|------|------------|--------|
| LLM の初期化 | `init_chat_model` | `プロバイダー:モデル名` 形式で統一的に指定できる |
| エージェントの作成 | `create_agent` | LangChain v1.0 の標準的な方法、ツールと組み合わせて拡張できる |
| エージェントの呼び出し | `agent.invoke()` | `{"messages": [...]}` 形式で入力を渡す |

**次の章（第3章）では、** `@tool` でカスタムツールを追加し、
記憶（Checkpointer）を組み込んでより賢いエージェントを構築します。